# Quick design of NSOLT by dictionary learning

Python counterpart of the MATLAB scripts in `examples/quickdesign`:

| Section | MATLAB script |
|---|---|
| [Design by a quasi-Newton method](#Design-by-a-quasi-Newton-method) | `main_quickdesign.m` |
| [Design by stochastic gradient descent](#Design-by-stochastic-gradient-descent) | `main_quickdesign_sgd.m` |
| [Pre-designed NSOLTs](#Pre-designed-NSOLTs) | `disp_quickdesign.m` |

As `saivdr.dictionary.nsoltx.design.NsoltDictionaryLearning`, the design
alternates

1. the sparse approximation of the training images by iterative hard
   thresholding (`saivdr.sparserep.IterativeHardThresholding`, tested to
   agree with MATLAB), and
2. the update of the NSOLT parameters with the coefficients fixed.

The update uses PyTorch optimizers with automatic differentiation —
L-BFGS in place of `fminunc` (quasi-Newton) and AdaGrad as
`NsoltDictionaryUpdateSgd` — so the designed NSOLTs are not identical to
the MATLAB ones. The NSOLTs designed in MATLAB are loaded in the last
section.

In [ ]:
import math
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import torch
from skimage import color, data

from saivdr.dcnn import atomic_imshow, copy_params_syn2ana, create_nsolt_2d, share_level_params
from saivdr.dcnn.matlab_compat import load_nsoltx
from saivdr.restoration import psnr
from saivdr.sparserep import IterativeHardThresholding

DATA = pathlib.Path("../data")
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    # The tensors are small; many threads only add overhead
    torch.set_num_threads(min(8, torch.get_num_threads()))
torch.set_default_dtype(torch.float64)
print("device:", device)

## Training image

A 128×128 grayscale crop of `skimage.data.astronaut()` in place of
`peppers.png`, cropped at `px = py = 128` as in `main_quickdesign.m`.

In [ ]:
gray = color.rgb2gray(data.astronaut())
trn = torch.from_numpy(gray[128:256, 128:256].copy()).reshape(1, 1, 128, 128).to(device)
plt.imshow(trn[0, 0].cpu(), cmap="gray", vmin=0, vmax=1)
plt.axis("off");

## Design by a quasi-Newton method

`main_quickdesign.m`: 4 levels, 2×2 decimation, 4+4 channels, polyphase
order 4+4, one vanishing moment, `NumberOfSparseCoefficients =
numel/8`, random initial angles of standard deviation π/6, and 8
iterations. As in nsoltx, all the levels share one filter bank
(`share_level_params`).

In [ ]:
torch.manual_seed(0)
cfg = dict(
    number_of_channels=(4, 4),
    decimation_factor=(2, 2),
    polyphase_order=(4, 4),
    number_of_vanishing_moments=1,
    device=device,
)
nLevels = 4
analyzer = share_level_params(create_nsolt_2d(mode="Analyzer", number_of_levels=nLevels, **cfg))
synthesizer = share_level_params(create_nsolt_2d(mode="Synthesizer", number_of_levels=nLevels, **cfg))
with torch.no_grad():
    for p in synthesizer.parameters():
        p.add_(torch.randn(p.shape, generator=torch.Generator().manual_seed(0)).to(p) * math.pi / 6)
copy_params_syn2ana(analyzer, synthesizer)
nCoefs = trn.numel() // 8
print("parameters:", sum(p.numel() for p in synthesizer.parameters()), "coefficients kept:", nCoefs)

In [ ]:
def design_step(analyzer, synthesizer, images, n_coefs, optimizer_factory, n_steps):
    # 1. Sparse approximation with the current NSOLT
    iht = IterativeHardThresholding(synthesizer, analyzer, number_of_sparse_coefficients=n_coefs)
    _, coefs = iht(images)
    # 2. Update of the synthesizer with the coefficients fixed
    optimizer = optimizer_factory(synthesizer.parameters())
    loss_fn = lambda: (synthesizer(*coefs) - images).pow(2).mean()  # noqa: E731
    optimizer_steps(optimizer, loss_fn, n_steps)
    copy_params_syn2ana(analyzer, synthesizer)
    with torch.no_grad():
        return float(loss_fn())


def optimizer_steps(optimizer, loss_fn, n_steps):
    def closure():
        optimizer.zero_grad()
        loss = loss_fn()
        loss.backward()
        return loss

    for _ in range(n_steps):
        optimizer.step(closure)


lbfgs = lambda params: torch.optim.LBFGS(params, max_iter=20, line_search_fn="strong_wolfe")  # noqa: E731
mses = []
for it in range(8):
    mses.append(design_step(analyzer, synthesizer, trn, nCoefs, lbfgs, n_steps=3))
    print(f"MSE ({it + 1}) = {mses[-1]:.4e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(range(1, len(mses) + 1), mses, "o-")
axes[0].set_xlabel("iteration")
axes[0].set_ylabel("MSE")
# One level of the shared filter bank, for the atomic images
syn1 = create_nsolt_2d(mode="Synthesizer", number_of_levels=1, **cfg)
for name, layer in syn1.layers.items():
    if hasattr(layer, "angles"):
        layer.set_angles(synthesizer.layers[name].angles.detach())
        layer.set_mus(synthesizer.layers[name].mus)
atomic_imshow(syn1, patch_size=(16, 16), ax=axes[1])
axes[1].set_title("Atomic images of the designed NSOLT");

In [ ]:
# Sparse approximation of the training image with the designed NSOLT
iht = IterativeHardThresholding(synthesizer, analyzer, number_of_sparse_coefficients=nCoefs)
approx, _ = iht(trn)
plt.imshow(approx[0, 0].clamp(0, 1).cpu(), cmap="gray", vmin=0, vmax=1)
plt.title(f"{nCoefs} coefficients: {psnr(approx.clamp(0, 1), trn):.2f} dB")
plt.axis("off");

## Design by stochastic gradient descent

`main_quickdesign_sgd.m`: 64 random 128×128 patches, each reduced by
1/2^(4−1) to 16×16 (a "virtual" 4-level tree), one level,
`NumberOfSparseCoefficients = numel/8`, and AdaGrad (η = 0.01,
ε = 1e-8) with 128 steps per iteration. Initial angles have standard
deviation 0.1.

In [ ]:
from skimage.transform import resize

rng = np.random.default_rng(0)
patches = []
for _ in range(64):
    py, px = rng.integers(0, gray.shape[0] - 128 + 1), rng.integers(0, gray.shape[1] - 128 + 1)
    patches.append(resize(gray[py : py + 128, px : px + 128], (16, 16), anti_aliasing=True))
subimgs = torch.from_numpy(np.stack(patches)).unsqueeze(1).to(device)  # 64 x 1 x 16 x 16

analyzer_sgd = create_nsolt_2d(mode="Analyzer", number_of_levels=1, **cfg)
synthesizer_sgd = create_nsolt_2d(mode="Synthesizer", number_of_levels=1, **cfg)
with torch.no_grad():
    for p in synthesizer_sgd.parameters():
        p.add_(torch.randn(p.shape, generator=torch.Generator().manual_seed(1)).to(p) * 0.1)
copy_params_syn2ana(analyzer_sgd, synthesizer_sgd)
nCoefsSgd = subimgs[0].numel() // 8

adagrad = lambda params: torch.optim.Adagrad(params, lr=1e-2, eps=1e-8)  # noqa: E731
mses_sgd = []
for it in range(8):
    mses_sgd.append(design_step(analyzer_sgd, synthesizer_sgd, subimgs, nCoefsSgd, adagrad, n_steps=128))
    print(f"MSE ({it + 1}) = {mses_sgd[-1]:.4e}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
atomic_imshow(synthesizer_sgd, patch_size=(16, 16), ax=ax)
ax.set_title("Atomic images of the NSOLT designed by SGD");

## Pre-designed NSOLTs

`disp_quickdesign.m` shows the NSOLTs designed in MATLAB and saved in
`examples/quickdesign/results`. They were exported for Python by

```matlab
q = load('examples/quickdesign/results/nsolt_d2x2_c4+4_o4+4_v1_l4_n2048_peppers128x128.mat');
saivdr_export_nsoltx(q.nsolt, 'nsoltx_quickdesign_peppers128.mat')
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ax, (file, title) in zip(
    axes,
    [
        ("nsoltx_quickdesign_peppers128.mat", "main_quickdesign.m (fminunc)"),
        ("nsoltx_quickdesign_peppers128_sgd.mat", "main_quickdesign_sgd.m (AdaGrad)"),
    ],
):
    syn = load_nsoltx(DATA / file, mode="Synthesizer", number_of_levels=1)
    atomic_imshow(syn, patch_size=(16, 16), ax=ax)
    ax.set_title(title)

In [ ]:
# Sparse approximation of the training image with the NSOLT designed in MATLAB
net = load_nsoltx(DATA / "nsoltx_quickdesign_peppers128.mat", number_of_levels=4, device=device)
iht = IterativeHardThresholding(net.synthesizer, net.analyzer, number_of_sparse_coefficients=nCoefs)
approx, _ = iht(trn)
print(f"PSNR with {nCoefs} coefficients: {psnr(approx.clamp(0, 1), trn):.2f} dB")